In [ ]:
# Setup

import re
import ctypes, json, random
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import string, time
from datasets import load_dataset
import textwrap


try:  # NixOS: make libcuda visible
    ctypes.CDLL("/run/opengl-driver/lib/libcuda.so.1", mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass

DEVICE, SEED = "cuda", 1
random.seed(SEED)

PAIRS      = {"en_de": ("en", "de"), "en_fr": ("en", "fr"), "en_it": ("en", "it")}
FILES      = {name: Path(f"./pairs_{name}_tokenised_clean.json") for name in PAIRS}   # next to the notebook
LABELER    = Path("rsd_head/labeler_ists.pt")                                       # from SynthData_iSTS_train.ipynb
LIMIT_DOCS = None            # None = whole corpus; start small to check timing and output
OUT        = Path("synthetic"); OUT.mkdir(exist_ok=True)

for p in [*FILES.values(), LABELER]:
    assert p.exists(), f"missing: {p}"
print("all inputs where found")

In [ ]:
# Define and check corpus

def load_docs(name, l1, l2):
    """{(year, doc): [sentence pairs sorted by sentence number]}"""
    docs = defaultdict(list)
    for p in json.load(open(FILES[name], encoding="utf-8")):
        assert p[l1] and p[l2], f"empty side in {p['id']}"
        docs[(p["year"], p["doc"])].append(p)
    return {k: sorted(v, key=lambda p: p["sentence"]) for k, v in sorted(docs.items())}

CORPUS = {name: load_docs(name, *langs) for name, langs in PAIRS.items()}

for name, (l1, l2) in PAIRS.items():
    docs  = CORPUS[name]
    sents = [p for d in docs.values() for p in d]
    words = {l: sum(len(p[l]) for p in sents) for l in (l1, l2)}
    punct = sum(lab == -1 for p in sents for l in (l1, l2) for _, lab in p[l]) / sum(words.values())
    longest = max(max(sum(len(p[l1]) for p in d), sum(len(p[l2]) for p in d)) for d in docs.values())
    print(f"{name}: {len(docs):,} docs | {len(sents):,} sentence pairs | "
          f"words {l1} {words[l1]:,} / {l2} {words[l2]:,} | punctuation {punct:.1%} | "
          f"longest doc {longest:,} words per side")

In [ ]:
# Pick encoder

MAX_WORDS = 2800                                   # max words per side per pass (≈ longest SwissGov doc)

lab     = torch.load(LABELER, weights_only=False)
ENCODER = lab["encoder"]
LAYER   = lab["layer"]
LEVELS  = np.asarray(lab["levels"])
SHARES  = lab["shares"]          # iSTS gold level shares per language -> per-corpus quantile cut-points
SMOOTH  = lab["smooth"]
DA      = lab["diffalign"]       # alpha, lo, hi, center

tok   = AutoTokenizer.from_pretrained(ENCODER)
model = AutoModel.from_pretrained(ENCODER, dtype=torch.bfloat16, device_map=DEVICE).eval()

H, HID = model.config.hidden_size, lab["hidden"]
heads = []
for state, mu, sd in lab["heads"]:
    h = torch.nn.Sequential(torch.nn.Linear(H, HID), torch.nn.GELU(), torch.nn.Linear(HID, 1)).to(DEVICE)
    h.load_state_dict(state)
    heads.append((h.eval(), mu.to(DEVICE), sd.to(DEVICE)))
CENTER = DA["center"].to(DEVICE)

print(f"{ENCODER} | layer {LAYER} | {len(heads)} heads | alpha {DA['alpha']} | smooth {SMOOTH} | "
      f"GPU {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

In [ ]:
# Punctuation score and Lable Rule


GOV_PUNCT, GOV_WORD = set(), set()
for r in load_dataset("ZurichNLP/SwissGov-RSD", split="validation"):       # tokens only; test untouched
    for toks, labs in ((r["text_a"].split(), r["labels_a"]), (r["text_b"].split(), r["labels_b"])):
        for t, l in zip(toks, labs):
            (GOV_PUNCT if l == -1 else GOV_WORD).add(t)

def is_punct(t):
    """SwissGov-RSD's definition: its own label if known, else -1 only for pure ASCII punctuation."""
    if t in GOV_PUNCT or t in GOV_WORD:
        return t in GOV_PUNCT
    return all(c in string.punctuation for c in t)

@torch.no_grad()
def encode(ctx_words, tgt_words):
    """'<other text>\\n\\n<target>' at layer LAYER -> one unit vector per target word (fp16 -> fp32, as in training)."""
    prefix = " ".join(ctx_words) + "\n\n"
    ends = len(prefix) + np.cumsum([len(w) + 1 for w in tgt_words]) - 1      # end char of each word
    enc  = tok(prefix + " ".join(tgt_words), return_tensors="pt", return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")[0]
    h    = model(**enc.to(DEVICE), output_hidden_states=True).hidden_states[LAYER][0].float()
    last = offs[:, 1] - 1
    keep = (offs[:, 1] > offs[:, 0]) & (last >= len(prefix))                  # target subwords only
    word = torch.as_tensor(np.searchsorted(ends, last[keep].numpy(), side="right"), device=DEVICE)
    E = torch.zeros(len(tgt_words), h.shape[-1], device=DEVICE).index_add_(0, word, h[keep.to(DEVICE)])
    return F.normalize(E, dim=-1).half().float()

def smooth(p, k, w):
    """Blend each score with the mean of its +-k neighbours (non-punctuation words of one text)."""
    if not k or len(p) == 0:
        return p
    nb = np.convolve(np.pad(p, k, mode="edge"), np.ones(2 * k + 1) / (2 * k + 1), mode="valid")
    return (1 - w) * p + w * nb

@torch.no_grad()
def side_scores(E, E_other, mask, mask_other):
    """Heads + DiffAlign blend + smoothing, as in the labeler notebook. Punctuation -> 0 (ignored later)."""
    out = np.zeros(len(mask))
    if not mask.any():
        return out
    Z = E[torch.as_tensor(mask, device=DEVICE)]
    head = torch.stack([torch.sigmoid(h((Z - mu) / sd)).squeeze(-1) for h, mu, sd in heads]).mean(0)
    if mask_other.any():
        A = F.normalize(Z - CENTER, dim=-1)
        B = F.normalize(E_other[torch.as_tensor(mask_other, device=DEVICE)] - CENTER, dim=-1)
        da = ((1 - (A @ B.T).max(1).values - DA["lo"]) / (DA["hi"] - DA["lo"])).clamp(0, 1)
    else:
        da = torch.full_like(head, 0.5)
    out[mask] = smooth(((1 - DA["alpha"]) * head + DA["alpha"] * da).cpu().numpy(), **SMOOTH)
    return out

def pair_scores(w1, w2):
    """Final labeler scores for both texts of a (chunk of a) document pair."""
    m1, m2 = np.array([not is_punct(t) for t in w1]), np.array([not is_punct(t) for t in w2])
    E1, E2 = encode(w2, w1), encode(w1, w2)
    return side_scores(E1, E2, m1, m2), side_scores(E2, E1, m2, m1)

# quick test on the first document of en_de

doc  = next(iter(CORPUS["en_de"].values()))
w_en = [t for p in doc for t, _ in p["en"]]
w_de = [t for p in doc for t, _ in p["de"]]
t0 = time.time()
d_en, d_de = pair_scores(w_en, w_de)
print(f"{time.time() - t0:.1f} s | en mean {d_en.mean():.3f} | de mean {d_de.mean():.3f}")

In [ ]:

def fit_cuts(scores, lang):
    """Cut-points so the level shares on this corpus equal the iSTS gold shares of `lang` (as in the labeler)."""
    return np.quantile(np.asarray(scores, float), np.cumsum(SHARES[lang])[:-1])

def prompt_scores(p, cuts):
    """Difference -> 6 levels -> paper scale (5 = equivalent ... 0 = no relation)."""
    level = LEVELS[np.searchsorted(cuts, np.asarray(p, float), side="right")]
    return (5 - np.rint(5 * level)).astype(int)

# test on the Cell 4 document (cut-points from this one document only, just for a look)
m_en = np.array([not is_punct(t) for t in w_en]); m_de = np.array([not is_punct(t) for t in w_de])
cuts = fit_cuts(np.concatenate([d_en[m_en], d_de[m_de]]), "de")
s = prompt_scores(d_en[:len(doc[0]["en"])], cuts)
print(" ".join(t if is_punct(t) else f"{t}[{x}]" for (t, _), x in zip(doc[0]["en"], s)))

In [ ]:
def chunks(sents, l1, l2):
    """Consecutive sentence pairs, split so that no side exceeds MAX_WORDS."""
    out, cur, n = [], [], 0
    for p in sents:
        k = max(len(p[l1]), len(p[l2]))
        if cur and n + k > MAX_WORDS:
            out.append(cur); cur, n = [], 0
        cur.append(p); n += k
    return out + [cur] if cur else out

def score_corpus(name, l1, l2):
    path = OUT / f"scores_{name}.jsonl"
    done = {r["id"] for r in map(json.loads, path.open())} if path.exists() else set()
    keys = list(CORPUS[name])[:LIMIT_DOCS] if LIMIT_DOCS else list(CORPUS[name])
    todo = [k for k in keys if CORPUS[name][k][0]["id"] not in done]
    with path.open("a", encoding="utf-8") as f:
        for key in tqdm(todo, desc=name):
            recs = []
            for group in chunks(CORPUS[name][key], l1, l2):
                w1 = [t for p in group for t, _ in p[l1]]
                w2 = [t for p in group for t, _ in p[l2]]
                a, b = pair_scores(w1, w2)
                d1 = np.split(a, np.cumsum([len(p[l1]) for p in group])[:-1])
                d2 = np.split(b, np.cumsum([len(p[l2]) for p in group])[:-1])
                for p, x, y in zip(group, d1, d2):
                    rec = {k: p[k] for k in ("id", "year", "doc", "sentence")}
                    for lang, raw in ((l1, x), (l2, y)):
                        rec[lang] = [[t, -1 if is_punct(t) else 0] for t, _ in p[lang]]   # tokens + punctuation flag
                        rec[f"{lang}_diff"] = np.round(raw, 4).tolist()                   # labeler score
                    recs.append(rec)
            f.write("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in recs)); f.flush()   # whole doc at once
            torch.cuda.empty_cache()
    print(f"{name}: {sum(1 for _ in path.open()):,} sentence pairs -> {path.name}")

for name, (l1, l2) in PAIRS.items():
    score_corpus(name, l1, l2)

In [ ]:
for name, (l1, l2) in PAIRS.items():
    recs = [json.loads(x) for x in (OUT / f"scores_{name}.jsonl").open(encoding="utf-8")]
    cuts = {}
    for lang in (l1, l2):                                             # per side: no language gets all the differences
        w = np.array([s for r in recs for (_, flag), s in zip(r[lang], r[f"{lang}_diff"]) if flag != -1])
        cuts[lang] = fit_cuts(w, l2)
    with (OUT / f"labels_{name}.jsonl").open("w", encoding="utf-8") as f:
        for r in recs:
            for lang in (l1, l2):
                ps = prompt_scores(r[f"{lang}_diff"], cuts[lang])
                r[lang] = [[t, -1 if flag == -1 else int(s)] for (t, flag), s in zip(r[lang], ps)]
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{name}: {len(recs):,} sentence pairs | cut-points " +
          " | ".join(f"{l} {np.round(c, 3)}" for l, c in cuts.items()) + f" -> labels_{name}.jsonl")

In [ ]:
# Inspect synthetic labels

N_EXAMPLES, WIDTH = 3, 110
ALL_LABELS = True                                       # False: only mark differences (score < 5)
GOLD_DIFF  = {"de": 12.7, "fr": 10.2, "it": 18.2}       # % different words in SwissGov-RSD dev, for reference

def load_labels(name):
    return [json.loads(x) for x in (OUT / f"labels_{name}.jsonl").open(encoding="utf-8")]

def show(rec, l1, l2, all_labels=ALL_LABELS):
    """Print a sentence pair with token labels: 5 ... 0, punctuation -1."""
    print(rec["id"])
    for lang in (l1, l2):
        text = " ".join(f"{t}[{s}]" if all_labels or s not in (5, -1) else t for t, s in rec[lang])
        print(textwrap.fill(text, WIDTH, initial_indent=f"  {lang}: ", subsequent_indent="      "))
    print()

for name, (l1, l2) in PAIRS.items():
    recs = load_labels(name)
    docs = len({(r["year"], r["doc"]) for r in recs})
    c = Counter(s for r in recs for lang in (l1, l2) for _, s in r[lang])
    words = sum(n for s, n in c.items() if s != -1)
    print("=" * WIDTH)
    print(f"{name}: {docs} docs | {len(recs):,} sentence pairs | {words:,} words | "
          f"different {100 * (1 - c[5] / words):.1f}% (SwissGov dev: {GOLD_DIFF[l2]}%)")
    print("  scores: " + " | ".join(f"{s}: {100 * c[s] / words:.1f}%" for s in range(5, -1, -1))
          + f" | punctuation: {c[-1]:,}")
    flagged = [r for r in recs if any(s not in (5, -1) for lang in (l1, l2) for _, s in r[lang])]
    print(f"  sentence pairs with at least one difference: {100 * len(flagged) / len(recs):.1f}%\n")
    for r in random.sample(recs, min(N_EXAMPLES, len(recs))):      # unbiased sample of all pairs
        show(r, l1, l2)

In [ ]:
# Check stored pairs

NUM = re.compile(r"\d+")

def suspicious(p, l1, l2):
    """Rough flags: very different length, or numbers on both sides with none in common."""
    a, b = p[f"{l1}_text"], p[f"{l2}_text"]
    ratio = len(b) / max(len(a), 1)
    na, nb = set(NUM.findall(a)), set(NUM.findall(b))
    flags = []
    if ratio > 2 or ratio < 0.5:
        flags.append(f"length x{ratio:.1f}")
    if na and nb and not na & nb:
        flags.append("numbers differ")
    return flags

for name, (l1, l2) in PAIRS.items():
    recs = [p for d in CORPUS[name].values() for p in d]
    flagged = [p for p in recs if suspicious(p, l1, l2)]
    print(f"{name}: {len(recs):,} pairs | suspicious {100 * len(flagged) / len(recs):.1f}%")

def show_pairs(name, year, doc, l1, l2, n=12):
    """First n stored pairs of one document, side by side."""
    for p in CORPUS[name][(year, doc)][:n]:
        f = suspicious(p, l1, l2)
        print(f"s{p['sentence']:03d} {'  <-- ' + ', '.join(f) if f else ''}")
        print(f"   {l1}: {p[f'{l1}_text']}\n   {l2}: {p[f'{l2}_text']}\n")

show_pairs("en_fr", "1999", 10, "en", "fr")

In [ ]:
# Create chat format

APERTUS     = "swiss-ai/Apertus-v1.5-8B"
MAX_TOKENS  = 16384          # must equal sequence_len in the Axolotl config
VALID_SHARE = 0.02           # held-out documents for eval loss during training
VERSIONS    = ("mixed",)

FEWSHOT_PROMPT_MIXED = """
Annotate a pair of either English-German, English-French or English-Italian input sentences. The goal is to label every word in each sentence regarding its semantic similarity to the words in the other sentence.

## Example 1:
Input sentence 1: ["Former Nazi death camp guard Demjanjuk dead at 91"]
Input sentence 2: ["John Demjanjuk, verurteilter Nazi-Todeslagerwächter, stirbt im Alter von 91 Jahren"]
Output: {"sentence1": [["Former", 1],["Nazi", 5],["death", 5],["camp", 5],["guard", 5],["Demjanjuk", 5],["dead", 5],["at", 5],["91", 5]],"sentence2": [["John", 0],["Demjanjuk", 5],[",", -1],["verurteilter", 1],["Nazi", 1],["-", -1],["Todeslagerwächter", 5],[",", -1],["stirbt", 5],["im", 4],["Alter", 4],["von", 4],["91", 5],["Jahren", 4]]}


## Example 2:
Input sentence 1: ["Saudis to permit women to compete in Olympics"]
Input sentence 2: ["Les femmes saoudiennes autorisées à participer aux Jeux olympiques"]
Output: {"sentence1": [["Saudis", 2], ["to", 1], ["permit", 1], ["women", 4], ["to", 0], ["compete", 3], ["in", 5], ["Olympics", 5]], "sentence2": [["Les", 4], ["femmes", 4], ["saoudiennes", 2], ["autorisées", 1], ["à", 3], ["participer", 1], ["aux", 5], ["Jeux", 5], ["olympiques", 5]]}


## Example 3:
Input sentence 1: ["6.8 quake strikes off Solomon Islands"]
Input sentence 2: ["Scossa di magnitudo 6,3 al largo delle Isole Salomone: USGS"]
Output: {"sentence1": [["6.8", 1], ["quake", 5], ["strikes", 0], ["off", 5], ["Solomon", 5], ["Islands", 5]], "sentence2": [["Scossa", 5], ["di", 3], ["magnitudo", 3], ["6", 5], [",", -1], ["3", 0], ["al", 5], ["largo", 5], ["delle", 5], ["Isole", 5], ["Salomone", 5], [":", -1], ["USGS", 0]]}

## Score scale

* **5:** Complete equivalence
* **3-4:** Very similar or closely similar in terms of semantics.
* **1–2** Slightly similar or somewhat similar.
* **0:** No relation (there is no word in the other sentence that is even slightly similar in terms of semantics).
* **-1:** Punctuation

## Other guidelines
* The two texts are given as normal running text. In the output, each text is listed item by item in a fixed order. Punctuation marks, hyphens and apostrophes are separate items, e.g. "Switzerland's" → "Switzerland", "'", "s" and "mid-Winter" → "mid", "-", "Winter".
* The items are fixed; your task is to give every item a score from the scale above, in the order listed.
* Output only the raw JSON, without any additional text.

## Input to Annotate

Sentence 1: {{ sentence1 }}
Sentence 2: {{ sentence2 }}

Respond with the JSON object.
""".strip()

def build_prompt(text1, text2):
    """Inputs exactly like the few-shot examples: a JSON list holding the running text as one string."""
    return (FEWSHOT_PROMPT_MIXED
            .replace("{{ sentence1 }}", json.dumps([text1], ensure_ascii=False))
            .replace("{{ sentence2 }}", json.dumps([text2], ensure_ascii=False)))

def answer(s1, s2):
    return json.dumps({"sentence1": s1, "sentence2": s2}, ensure_ascii=False)   # keys as in SwissGov-RSD

def chat_record(t1, t2, s1, s2):
    return {"messages": [{"role": "user", "content": build_prompt(t1, t2)},
                         {"role": "assistant", "content": answer(s1, s2)}]}

apertus_tok = AutoTokenizer.from_pretrained(APERTUS)

def n_tokens(messages):
    """Length of the rendered chat. (apply_chat_template(tokenize=True) returns a dict in recent
    transformers, so len() of it was always 2 and the MAX_TOKENS filter never fired.)"""
    text = apertus_tok.apply_chat_template(messages, tokenize=False)
    return len(apertus_tok(text, add_special_tokens=False)["input_ids"])

SEP_CHARS = ("'", "’", "-")
SEP = re.compile(r"(['’\-])")

def split_items(items):
    """Hyphens and apostrophes as separate items (-1), as the prompt says: Switzerland's -> Switzerland ' s.
    Both word parts keep the word's score."""
    out = []
    for t, s in items:
        for part in SEP.split(t):
            if part:
                out.append([part, -1 if part in SEP_CHARS else s])
    return out

# documents

DOCS, skipped = [], Counter()
for name, (l1, l2) in PAIRS.items():
    docs = defaultdict(list)
    for r in load_labels(name):
        docs[(r["year"], r["doc"])].append(r)
    for (year, doc), recs in docs.items():
        if (year, doc) in globals().get("CONTAMINATED", set()):
            skipped["overlap with SwissGov"] += 1; continue
        recs.sort(key=lambda r: r["sentence"])
        # target items: drop empty tokens, remove whitespace inside tokens (e.g. "550\xa0000"), split - and '
        s1 = split_items([["".join(t.split()), s] for r in recs for t, s in r[l1] if t.strip()])
        s2 = split_items([["".join(t.split()), s] for r in recs for t, s in r[l2] if t.strip()])
        if max(len(s1), len(s2)) > MAX_WORDS:                            # was split for labeling
            skipped["> MAX_WORDS"] += 1; continue
        sents = CORPUS[name][(year, doc)]                                 # original running text per sentence
        t1 = " ".join(p[f"{l1}_text"].strip() for p in sents)
        t2 = " ".join(p[f"{l2}_text"].strip() for p in sents)
        DOCS.append(dict(id=f"{name}_{year}_d{doc:04d}", group=f"{year}_d{doc:04d}", l2=l2,
                         t1=t1, t2=t2, s1=s1, s2=s2))
print(f"{len(DOCS):,} documents | skipped {dict(skipped)}")

# document split; all language versions of a document on the same side

groups = sorted({d["group"] for d in DOCS})
rng = random.Random(SEED)
valid_groups = set(rng.sample(groups, max(1, int(VALID_SHARE * len(groups)))))
rng.shuffle(DOCS)

for version in VERSIONS:
    lens, too_long = [], 0
    files = {s: (OUT / f"{s}_{version}.jsonl").open("w", encoding="utf-8") for s in ("train", "valid")}
    for d in tqdm(DOCS, desc=version):
        ex = chat_record(d["t1"], d["t2"], d["s1"], d["s2"])
        n = n_tokens(ex["messages"])
        if n > MAX_TOKENS:
            too_long += 1; continue
        lens.append(n)
        split = "valid" if d["group"] in valid_groups else "train"
        files[split].write(json.dumps({"messages": ex["messages"], "id": d["id"]}, ensure_ascii=False) + "\n")
    for f in files.values():
        f.close()
    lens = np.array(lens)
    print(f"{version}: train {sum(1 for _ in (OUT / f'train_{version}.jsonl').open()):,} | "
          f"valid {sum(1 for _ in (OUT / f'valid_{version}.jsonl').open()):,} | skipped > MAX_TOKENS {too_long} | "
          f"tokens median {int(np.median(lens)):,} p95 {int(np.quantile(lens, .95)):,} max {lens.max():,}")

# the prompt for one short document
d = min(DOCS, key=lambda d: len(d["s1"]) + len(d["s2"]))
print("\n" + build_prompt(d["t1"], d["t2"]))

In [ ]:
# Check validity

VALID_LABELS = {-1, 0, 1, 2, 3, 4, 5}

def parse(ex):
    """Running texts from the prompt (JSON list with one string) + the target JSON."""
    user, out = ex["messages"][0]["content"], json.loads(ex["messages"][1]["content"])
    tail = user.split("## Input to Annotate\n\n")[-1]
    t1 = json.loads(tail.split("Sentence 1: ", 1)[1].split("\nSentence 2: ", 1)[0])[0]
    t2 = json.loads(tail.split("Sentence 2: ", 1)[1].split("\n\nRespond", 1)[0])[0]
    return user, t1, t2, out

nospace = lambda s: "".join(s.split())

for version in VERSIONS:
    for split in ("train", "valid"):
        rows = [json.loads(x) for x in (OUT / f"{split}_{version}.jsonl").open(encoding="utf-8")]
        problems, labels, per_pair = Counter(), Counter(), Counter()
        for ex in rows:
            per_pair[ex["id"][:5]] += 1
            if [m["role"] for m in ex["messages"]] != ["user", "assistant"]:
                problems["wrong roles"] += 1; continue
            try:
                user, t1, t2, out = parse(ex)
            except Exception:
                problems["not parseable"] += 1; continue
            if not user.startswith("Annotate a pair of either English-German"):
                problems["wrong template"] += 1
            for text, key in ((t1, "sentence1"), (t2, "sentence2")):
                if nospace("".join(t for t, _ in out[key])) != nospace(text):
                    problems[f"{key}: items do not spell the input text"] += 1
                for t, s in out[key]:
                    labels[s] += 1
                    if s not in VALID_LABELS: problems["label out of range"] += 1
                    if (s == -1) != (is_punct(t) or t in SEP_CHARS): problems["-1 does not match punctuation rule"] += 1
        words = sum(n for s, n in labels.items() if s != -1)
        print(f"=== {version} / {split}: {len(rows):,} examples | "
              + " | ".join(f"{k} {v:,}" for k, v in sorted(per_pair.items())))
        print("  labels: " + " | ".join(f"{s}: {100 * labels[s] / words:.1f}%" for s in range(5, -1, -1))
              + f" | punctuation {labels[-1]:,}")
        print("  problems:", dict(problems) if problems else "none")

# one short example, as Apertus sees it in training
ex = min(rows, key=lambda e: len(e["messages"][1]["content"]))
user, t1, t2, out = parse(ex)
print(f"\n=== {ex['id']} ({len(out['sentence1'])} + {len(out['sentence2'])} items) ===")
print(user[user.index("## Input to Annotate"):])
for key in ("sentence1", "sentence2"):
    print(f"  {key}: " + " ".join(f"{t}[{s}]" for t, s in out[key]))
print("\n--- rendered with the Apertus chat template (start) ---")
print(apertus_tok.apply_chat_template(ex["messages"], tokenize=False)[:500])

In [ ]:
for name, (l1, l2) in PAIRS.items():
    recs = load_labels(name)
    for lang in (l1, l2):
        s = np.array([x for r in recs for _, x in r[lang] if x != -1])
        print(f"{name} {lang}: different {100 * np.mean(s < 5):.1f}% | level 0 {100 * np.mean(s == 0):.1f}%")

In [ ]:
    cuts = {}
    for lang in (l1, l2):                                             # per side: no language gets all the differences
        w = np.array([s for r in recs for (_, flag), s in zip(r[lang], r[f"{lang}_diff"]) if flag != -1])
        cuts[lang] = fit_cuts(w, l2)
    with (OUT / f"labels_{name}.jsonl").open("w", encoding="utf-8") as f:
        for r in recs:
            for lang in (l1, l2):
                ps = prompt_scores(r[f"{lang}_diff"], cuts[lang])
                r[lang] = [[t, -1 if flag == -1 else int(s)] for (t, flag), s in zip(r[lang], ps)]
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{name}: {len(recs):,} sentence pairs | cut-points " +
          " | ".join(f"{l} {np.round(c, 3)}" for l, c in cuts.items()) + f" -> labels_{name}.jsonl")